In [3]:
import os
from dataset import  CDCR_Dataset
from sentence_transformers import SentenceTransformer, util
import statistics
import torch
import json

from collections import defaultdict

#Load data
ecb_paths = ['../CDCR/data/ECBplus-prep/output_data/train']
cdcr_train = CDCR_Dataset(name="cdcr_train", data_path=ecb_paths,context_method="hsu")


def remove_special_token(token_list):
    return [token for token in token_list if token not in ('[E]','[/E]')]

# Load SBERT model
model = SentenceTransformer('all-mpnet-base-v2')

c:\ProgramData\miniconda3\envs\mbib\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Here I try to create a list of sentences by topic

In [4]:
dict_sentences_by_topic = defaultdict(dict)
#Create dictionary of sentences by topic
# To compare sentence similarity by these groups
for doc_key, doc in cdcr_train.sentences.items():
    topic_num = doc_key.split("/")[0]
    for index, sentence in doc.items():
        sentence_str = ' '.join(sentence.values()) 
        sentence_full_key = f"{doc_key}/{index}"
        dict_sentences_by_topic[topic_num].update({sentence_full_key:sentence_str})

In [5]:
dict_sentences_by_topic.keys()

dict_keys(['1', '10', '11', '13', '14', '16', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '36', '37', '38', '39', '4', '41', '43', '44', '6', '7', '8', '9'])

In [6]:
# #TAKES TOO LONG, LOAD RESULTS INSTEAD

# #For each topic
# dict_sentences_pair_by_topic = {}
# for topic,sentences_dict in dict_sentences_by_topic.items():
#     print(f'current topic {topic}')
#     sentences = list(sentences_dict.values())
    
#     #Compute similarity
#     if not sentences:
#         continue
#     sentence_embeddings = model.encode(sentences, convert_to_tensor=True)
#     cosine_scores = util.cos_sim(sentence_embeddings, sentence_embeddings)

#     list_of_keys = list(sentences_dict.keys())
#     # Get the indices of the lower triangular elements
#     #Validated that the result is the combination
#     indices = torch.tril(torch.ones_like(cosine_scores), diagonal=-1).nonzero()

#     # Create the dictionary using the indices
#     # Also 
#     lower_triangle_dict = {}

#     for i, j in indices:
#         key_sentence1 = list_of_keys[i.item()]
#         key_sentence2 = list_of_keys[j.item()]
#         key_pair = (key_sentence1, key_sentence2)
#         #Ignore sentences without mentions
#         if not cdcr_train.mentions_by_sentence[key_sentence1] or not cdcr_train.mentions_by_sentence[key_sentence2]:
#             continue

#         similarity_score = round(cosine_scores[i, j].item(), 3)
#         if similarity_score<1: #ignores those sentences with similarity=1. Mostly exact citations
#             lower_triangle_dict[key_pair] = similarity_score

#     dict_sentences_pair_by_topic[topic] = lower_triangle_dict


In [7]:
# # Export results
# # Convert tuple keys to strings
# def convert_keys_to_strings(d):
#     new_dict = {}
#     for key, value in d.items():
#         if isinstance(key, tuple):
#             key = '-'.join(map(str, key))  # Convert tuple to a string
#         if isinstance(value, dict):
#             value = convert_keys_to_strings(value)  # Recursively convert nested dictionaries
#         new_dict[key] = value
#     return new_dict

# nested_dict_strings = convert_keys_to_strings(dict_sentences_pair_by_topic)


# json_file_path = 'sentence_similarities.json'

# # Export the nested dictionary to JSON
# with open(json_file_path, 'w') as json_file:
#     json.dump(nested_dict_strings, json_file, indent=4)

# print(f'Nested dictionary has been exported to {json_file_path}.')

In [ ]:
with open(json_file_path_np, 'r') as json_file:
    negative_pairs = json.load(json_file)

In [13]:
json_file_path = 'sentence_similarities.json'
with open(json_file_path, 'r') as json_file:
    nested_dict_strings = json.load(json_file)

In [14]:
dict_sentences_pair_by_topic = {}

for key, inner_dict in nested_dict_strings.items():
    new_inner_dict = {tuple(k.split('-')): v for k, v in inner_dict.items()}
    dict_sentences_pair_by_topic[key] = new_inner_dict

In [16]:
## TODO validate similarity score for a few sentences

## Move from sentences to mentions

In [196]:
from itertools import product
import numpy as np

dict_positive_pairs_by_topic = defaultdict(dict)
dict_negative_pairs_by_topic = defaultdict(dict)

partial_type = 'entity'

# Havent considered creating mentions only from similar or dissimilar sentences
for topic, sentence_pairs in dict_sentences_pair_by_topic.items():
    # Regroup pairs of sentences in similar and dissimilar.
    print(topic)
    if not sentence_pairs: 
        continue
    topic_similarity_score_median = statistics.median(list(sentence_pairs.values()))
    for (sentence_key1, sentence_key2), score in sentence_pairs.items():
        list1 = cdcr_train.mentions_by_sentence[sentence_key1]
        list2 = cdcr_train.mentions_by_sentence[sentence_key2]

        #Check all the combinations of menitions for each pair of sentences
        for mention_key1,mention_key2 in product(list1, list2):
            #Still need to filter by entity type
            mention1 = cdcr_train.mentions[mention_key1]
            mention2 = cdcr_train.mentions[mention_key2]
            
            if mention1.type!= partial_type or mention2.type!= partial_type:
                continue

            same_coref_chain = mention1.coref_chain == mention2.coref_chain
            if same_coref_chain:
                dict_positive_pairs_by_topic[topic].update({(mention_key1,mention_key2):score})
            if not same_coref_chain: 
                dict_negative_pairs_by_topic[topic].update({(mention_key1,mention_key2):score})
        
        

    
    positive_pairs_score = list(dict_positive_pairs_by_topic[topic].values())
    positive_pairs_score_median = statistics.median(positive_pairs_score)
    #Downsample negative pairs based on positive pairs score median
    third_quartile = np.percentile(positive_pairs_score, 75)

    dict_negative_pairs_by_topic[topic] = {key: value for key, value in dict_negative_pairs_by_topic[topic].items() if value > third_quartile}
    #Downsample positive pairs based on topic's median
    dict_positive_pairs_by_topic[topic] = {key: value for key, value in dict_positive_pairs_by_topic[topic].items() if value < topic_similarity_score_median}



1
10
11
13
14
16
18
19
20
21
22
23
24
25
26
27
28
29
3
30
31
32
33
34
36
37
38
39
4
41
43
44
6
7
8
9


In [202]:
suma = 0
for topic in dict_negative_pairs_by_topic:
    suma+=len(dict_negative_pairs_by_topic[topic])
print(suma)

85996


In [203]:
85000/4

21250.0

In [160]:
len(dict_negative_pairs_by_topic['10'])

111

In [126]:
topic_similarity_score_median

0.398

In [135]:
positive_pairs_score = list(dict_positive_pairs_by_topic['1'].values())
positive_pairs_score_median = statistics.median(positive_pairs_score)
#Downsample negative pairs based on positive pairs score median

filtered_dict = {key: value for key, value in dict_negative_pairs_by_topic['1'].items() if value > positive_pairs_score_median}
#Downsample positive pairs based on topic's median
filtered_dict_pos = {key: value for key, value in dict_positive_pairs_by_topic['1'].items() if value < topic_similarity_score_median}




In [137]:
filtered_dict_pos

{('1_10ecb_ACT16236402809085484_3_8_JrWN',
  '1_1ecb_ACT16236402809085484_1_35_UPVY'): 0.337,
 ('1_10ecb_ACT16236402809085484_3_21_2Wws',
  '1_1ecb_ACT16236402809085484_1_35_UPVY'): 0.337,
 ('1_11ecb_ACT16236402809085484_0_3_kxSd',
  '1_1ecb_ACT16236402809085484_2_11_dPhp'): 0.288,
 ('1_11ecb_ACT16235311629112331_0_14_PL2z',
  '1_1ecb_ACT16235311629112331_2_7_LgYu'): 0.288,
 ('1_11ecb_ACT16236402809085484_0_3_kxSd',
  '1_10ecb_ACT16236402809085484_3_8_JrWN'): 0.144,
 ('1_11ecb_ACT16236402809085484_0_3_kxSd',
  '1_10ecb_ACT16236402809085484_3_21_2Wws'): 0.144,
 ('1_11ecb_ACT16236402809085484_2_4_Ychs',
  '1_10ecb_ACT16236402809085484_3_8_JrWN'): 0.379,
 ('1_11ecb_ACT16236402809085484_2_4_Ychs',
  '1_10ecb_ACT16236402809085484_3_21_2Wws'): 0.379,
 ('1_13ecb_ACT16236402809085484_1_16_VVnJ',
  '1_1ecb_ACT16236402809085484_2_11_dPhp'): 0.269,
 ('1_13ecb_ACT16236402809085484_1_16_VVnJ',
  '1_10ecb_ACT16236402809085484_3_8_JrWN'): 0.163,
 ('1_13ecb_ACT16236402809085484_1_16_VVnJ',
  '1_10ecb_

In [ ]:
negative_pairs_scores = list(dict_negative_pairs_by_topic['1'].values())
sampled_negative_pairs = [score for score in negative_pairs_scores if score >positive_pairs_score_median]
sampled_positive_pairs = [score for score in positive_pairs_score if score >positive_pairs_score_median]
positive_pairs_score_median
dict_positive_pairs_by_topic
negative_pairs_scores

In [120]:

statistics.median(negative_pairs_scores)


0.554

In [123]:

print(len(negative_pairs_scores))
print(len(sampled_negative_pairs))

9620
7786


In [124]:
negative_pairs_scores = sampled_negative_pairs
statistics.median(negative_pairs_scores)

0.745

In [77]:
positive_pairs_scores = list(dict_mentions_by_topic['1'].values())

In [78]:
import statistics
statistics.median(positive_pairs_scores)

0.568

In [87]:
original_median = list(dict_sentences_pair_by_topic['1'].values())
statistics.median(original_median)

0.398

In [91]:
len([score for score in positive_pairs_scores if score < 0.398])

445

In [89]:
len(positive_pairs_scores)

1569

In [62]:
cdcr_train.mentions[key1].type

'event'

In [ ]:
#Still need to validate whether I am getting the right value of cosine similarity of two random pairs
# of sentences

In [64]:
len(dict_sentences_pair_by_topic['1'])

55611

In [76]:

topic_similarity_values = list(dict_sentences_pair_by_topic['1'].values())

In [100]:
max(topic_similarity_values)
for k,v in dict_sentences_pair_by_topic['1'].items():
    if v <0:
        print(k)

('1/1ecb/11/3', '1/1ecb/10/4')
('1/1ecb/11/7', '1/1ecb/11/3')
('1/1ecb/11/10', '1/1ecb/11/4')
('1/1ecb/11/14', '1/1ecb/1/1')
('1/1ecb/11/14', '1/1ecb/10/0')
('1/1ecb/11/14', '1/1ecb/11/0')
('1/1ecb/11/14', '1/1ecb/11/2')
('1/1ecb/11/14', '1/1ecb/11/6')
('1/1ecb/11/14', '1/1ecb/11/12')
('1/1ecb/11/15', '1/1ecb/10/1')
('1/1ecb/11/15', '1/1ecb/10/2')
('1/1ecb/11/15', '1/1ecb/11/3')
('1/1ecb/11/15', '1/1ecb/11/4')
('1/1ecb/11/15', '1/1ecb/11/8')
('1/1ecb/11/15', '1/1ecb/11/12')
('1/1ecb/11/16', '1/1ecb/11/0')
('1/1ecb/11/16', '1/1ecb/11/4')
('1/1ecb/11/16', '1/1ecb/11/8')
('1/1ecb/11/17', '1/1ecb/10/2')
('1/1ecb/11/17', '1/1ecb/10/4')
('1/1ecb/11/18', '1/1ecb/10/2')
('1/1ecb/11/18', '1/1ecb/11/0')
('1/1ecb/11/18', '1/1ecb/11/4')
('1/1ecb/11/19', '1/1ecb/10/4')
('1/1ecb/11/19', '1/1ecb/11/4')
('1/1ecb/11/19', '1/1ecb/11/12')
('1/1ecb/11/20', '1/1ecb/1/1')
('1/1ecb/11/20', '1/1ecb/10/2')
('1/1ecb/11/20', '1/1ecb/11/0')
('1/1ecb/11/20', '1/1ecb/11/4')
('1/1ecb/11/20', '1/1ecb/11/12')
('1/1ecb

In [104]:
dict_sentences_by_topic['1']['1/1ecbplus/9/5']

'Lohan had checked into Morningside Recovery Center Thursday morning , but then left because “ she was not happy with the place , ” her father said .'

In [103]:
dict_sentences_by_topic['1']['1/1ecbplus/8/2']

'Published May 03 , 2013'

In [ ]:
# I expect a considerable reduction of positive pairs given that the algorithm yields very dissimilar 
# sentences.

In [77]:
import statistics
statistics.median(topic_similarity_values)

0.172

In [ ]:
#I should have something like 
# '1/1ecb/1/0' - '1/1ecb/1/1'
# '1/1ecb/1/0' - '1/1ecb/1/2'
# And so forth.
# this I organize by similarity ignoring same sentences.
# Then for similar pairs I extract negative samples, from dissimilar pairs I extract positive samples
# I should create something like mentions_by_sentence in dataset.py

# I should do:
# "conll_doc_key": "12/12ecb/10"
# "conll_sen_key": "12/12ecb/10/{sent_id}"
# Then I have to extract mentions from the chosen sentences
# If it is strict lower triangle equal indices should not exist
# I can test some sentences to see if the indexing works well


In [ ]:
# I need to calculate the median from the each topic in the dictionary.
# I am not getting the conll-doc from the pairs the way I am creating the dictionary. Or maybe
# I am?

In [ ]:
# The next problem I was thinking of was the combination of all the mentions from one sentence with
# All the mentions from another one

In [60]:
cdcr_train.mentions['1_10ecb_TIM16236441119149059_3_11_YJuG'].sent_id

3

In [8]:
cdcr_train.sentences['1/1ecb/1'][0]

{0: 'Another',
 1: 'day',
 2: 'in',
 3: 'Hollywood',
 4: ';',
 5: 'another',
 6: 'star',
 7: 'in',
 8: 'rehab',
 9: '.'}

In [18]:
topic_list = set([mention.topic_id for _,mention in cdcr_train.mentions.items()])
mention_types = set([mention.type for _,mention in cdcr_train.mentions.items()])

In [19]:
dict_positive_pairs = {mention_type: {} for mention_type in mention_types}
dict_negative_pairs = {mention_type: {} for mention_type in mention_types}


for topic in topic_list:
    for mention_type in mention_types:
        keys_current_topic= cdcr_train.mentions_by_topics[topic]
        #filter by mention type
        mentions_sentences_current_topic= {k: remove_special_token(mention.new_context[1])  for k,mention in cdcr_train.mentions.items() if k in keys_current_topic and mention.type==mention_type}
        if not mentions_sentences_current_topic:
            continue
        #concatenate tokens
        sentences_list = [" ".join(sentence) for sentence in list(mentions_sentences_current_topic.values())]
        print("topic",topic,"mention_type",mention_type,len(sentences_list))
        #Compute similarity
        sentence_embeddings = model.encode(sentences_list, convert_to_tensor=True)
        cosine_scores = util.cos_sim(sentence_embeddings, sentence_embeddings)

        list_of_keys = list(mentions_sentences_current_topic.keys())
        # Get the indices of the lower triangular elements
        indices = torch.tril(torch.ones_like(cosine_scores), diagonal=-1).nonzero()

        # Create the dictionary using the indices
        lower_triangle_dict = {(i.item(), j.item()): cosine_scores[i, j].item() for i, j in indices}

        #classify pairs in: pairs from the same sentence, positives and negatives
  
        positive_pairs = {}
        negative_pairs = {}
        for k,v in lower_triangle_dict.items():
            row, col = k
            mention1 = cdcr_train.mentions[list_of_keys[row]]
            mention2 = cdcr_train.mentions[list_of_keys[col]]
            pair_key = f"{mention1.mention_id}###{mention2.mention_id}"
            #Ignore mentions from the same sentence
            if mention1.conll_doc_key==mention2.conll_doc_key and mention1.sent_id==mention2.sent_id:
                pass
            elif mention1.coref_chain == mention2.coref_chain:
                positive_pairs[pair_key]=v
            else: 
                negative_pairs[pair_key] = v
        
        # Concatenate positive_pairs and negative_pairs to the respective dictionaries
        dict_positive_pairs[mention_type].update(positive_pairs)
        dict_negative_pairs[mention_type].update(negative_pairs)
        


topic 19 mention_type event 128
topic 19 mention_type entity 191
topic 20 mention_type event 92
topic 20 mention_type entity 145
topic 28 mention_type event 176
topic 28 mention_type entity 327
topic 14 mention_type event 66
topic 14 mention_type entity 97
topic 32 mention_type event 108
topic 32 mention_type entity 227
topic 3 mention_type event 97
topic 3 mention_type entity 239
topic 30 mention_type event 169
topic 30 mention_type entity 420
topic 9 mention_type event 53
topic 9 mention_type entity 167
topic 34 mention_type event 79
topic 34 mention_type entity 318
topic 16 mention_type event 60
topic 16 mention_type entity 133
topic 24 mention_type event 99
topic 24 mention_type entity 225
topic 22 mention_type event 134
topic 22 mention_type entity 226
topic 29 mention_type event 146
topic 29 mention_type entity 315
topic 27 mention_type event 143
topic 27 mention_type entity 416
topic 33 mention_type event 144
topic 33 mention_type entity 297
topic 10 mention_type event 45
topic 

In [21]:
import json

json_file_path = "positive_pairs.json"

# Export dictionary to JSON file
with open(json_file_path, 'w') as json_file:
    json.dump(dict_positive_pairs, json_file, indent=4)

json_file_path = "negative_pairs.json"

# Export dictionary to JSON file
with open(json_file_path, 'w') as json_file:
    json.dump(dict_negative_pairs, json_file, indent=4)